In [ ]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

In [ ]:
def test_data_loader():
    print("==================================================")
    print("🚀 QuantDataLoader 테스트를 시작합니다...")
    print("==================================================\n")
    
    # 1. 로더 인스턴스 생성
    try:
        print("[테스트 1] 로더 인스턴스화 및 환경변수 확인")
        loader = QuantDataLoader(use_cache=True)
        print("✅ 성공: DART API 키 및 로더 초기화 완료!\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")
        return

    # 2. 유니버스 로드 테스트 (Point-in-Time)
    test_date = date(2023, 7, 24)
    print(f"[테스트 2] KOSPI 유니버스 데이터 로드 ({test_date})")
    try:
        universe_df = loader.get_kospi_universe(test_date)
        print(f"✅ 성공: 총 {len(universe_df)}개 종목 로드 완료!")
        print("-" * 50)
        display(universe_df.head())
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    # 3. DART 재무제표 파싱 테스트
    ticker_to_test = '005930'
    target_year = 2023
    print(f"[테스트 3] {ticker_to_test} {target_year}년 사업보고서(11011) 파싱")
    try:
        financials = loader.parse_standardized_financials(ticker_to_test, target_year, '11011')
        print(f"✅ 성공: 재무 데이터 표준화 완료!")
        print("-" * 50)
        for key, value in financials.items():
            if pd.isna(value):
                print(f"{key:>20} : NaN")
            else:
                print(f"{key:>20} : {value:,.0f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    print("==================================================")
    print("🎯 모든 테스트가 종료되었습니다.")
    print("==================================================")

# 테스트 실행
test_data_loader()

In [ ]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

def verify_new_features():
    print("==================================================")
    print("🚀 QuantDataLoader 신규 기능 검증을 시작합니다...")
    print("==================================================\n")
    
    try:
        loader = QuantDataLoader(use_cache=True)
    except Exception as e:
        print(f"❌ 초기화 실패: {e}")
        return

    # ---------------------------------------------------------
    # 검증 1: 시계열 주가/거래량 데이터 (OHLCV) 및 캐싱
    # ---------------------------------------------------------
    print("[검증 1] get_historical_ohlcv 작동 확인")
    ticker = '005930'
    start = date(2025, 1, 1)
    end = date(2025, 6, 30)
    
    try:
        ohlcv_df = loader.get_historical_ohlcv(ticker, start, end)
        if ohlcv_df is not None and not ohlcv_df.empty:
            print(f"✅ 성공: {start} ~ {end} 시계열 데이터 {len(ohlcv_df)}일치 로드 완료")
            print(ohlcv_df[['Close', 'Volume']].head(3).to_string())
        else:
            print("❌ 실패: 데이터가 비어 있습니다.")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 2: 확장된 계정과목 매핑 확인 (자산, 부채, 자본 등)
    # ---------------------------------------------------------
    print("[검증 2] 확장된 재무제표 계정 파싱 확인 (2025년 사업보고서)")
    try:
        fin_annual = loader.parse_standardized_financials(ticker, 2025, '11011')
        keys_to_check = ['total_assets', 'total_liabilities', 'total_equity', 'interest_expense']
        
        missing = [k for k in keys_to_check if pd.isna(fin_annual.get(k, float('nan')))]
        if not missing:
            print("✅ 성공: 자산/부채/자본/이자비용 모두 정상 파싱 완료")
            for k in keys_to_check:
                print(f"   - {k}: {fin_annual[k]:,.0f}")
        else:
            print(f"⚠️ 주의: 다음 계정 누락 (정상일 수도 있음) -> {missing}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 3: 분기 단독값 차분(Isolation) 로직 확인
    # ---------------------------------------------------------
    print("[검증 3] get_isolated_quarterly_financials 차분 로직 확인")
    try:
        # 1분기(누적)와 2분기(차분)의 매출액 비교
        q1_data = loader.get_isolated_quarterly_financials(ticker, 2025, 1)
        q2_isolated = loader.get_isolated_quarterly_financials(ticker, 2025, 2)
        
        print("✅ 성공: 1분기 및 2분기(단독) 데이터 추출 완료")
        print(f"   - 1Q 매출액 (누적=단독) : {q1_data.get('revenue', 0):,.0f}")
        print(f"   - 2Q 매출액 (차분 적용) : {q2_isolated.get('revenue', 0):,.0f}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 4: DART API Rate Limit 카운터 작동 확인
    # ---------------------------------------------------------
    print("[검증 4] DART API 카운터 및 Rate Limit 방어벽 확인")
    try:
        current_calls = loader.dart_call_count
        print(f"✅ 성공: 현재 세션 API 호출 횟수 정상 트래킹 중 -> {current_calls}회")
        
        # 임의로 한도를 초과시켜 방어벽 테스트
        loader.dart_daily_limit = current_calls  
        
        # 💡 API를 강제로 호출하도록 일시적으로 캐시 기능 비활성화
        loader.use_cache = False 
        
        try:
            loader.get_financial_statements(ticker, 2024, '11011')
            print("❌ 실패: 한도 초과 상황에서 Exception이 발생하지 않고 통과됨!")
        except Exception as limit_err:
            print(f"✅ 성공: 방어벽 정상 작동 확인 -> {limit_err}")
            
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("==================================================\n")


if __name__ == "__main__":
    verify_new_features()

In [ ]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

print("==================================================")
print("🚀 [검증] 공시 시차(Disclosure Lag) 및 미래참조 방지 테스트")
print("==================================================\n")

loader = QuantDataLoader(use_cache=True)
ticker = '005930' # 삼성전자
year = 2025
quarter = 1

# ---------------------------------------------------------
# 테스트 1: 공시 이전 시점 (Look-ahead bias 발생 가능 시점)
# 1분기 보고서 제출 마감일(보통 5월 15일) 이전인 '4월 30일' 기준
# ---------------------------------------------------------
base_date_before_release = date(2025, 4, 30)
print(f"▶️ 테스트 1: 기준일 = {base_date_before_release} (1Q 실적발표 전)")
data_before = loader.get_isolated_quarterly_financials(ticker, year, quarter, base_date=base_date_before_release)

if pd.isna(data_before.get('revenue')):
    print("✅ 성공: 아직 공시되지 않은 미래의 데이터를 정확히 차단하여 NaN을 반환했습니다.")
else:
    print(f"❌ 실패: 공시 전인데 미래 데이터를 가져왔습니다! 매출액: {data_before.get('revenue')}")

print("-" * 50)

# ---------------------------------------------------------
# 테스트 2: 공시 이후 시점 (정상적인 데이터 수집)
# 1분기 보고서 제출 마감일 이후인 '6월 1일' 기준
# ---------------------------------------------------------
base_date_after_release = date(2025, 6, 1)
print(f"\n▶️ 테스트 2: 기준일 = {base_date_after_release} (1Q 실적발표 후)")
data_after = loader.get_isolated_quarterly_financials(ticker, year, quarter, base_date=base_date_after_release)

if pd.notna(data_after.get('revenue')):
    print(f"✅ 성공: 공시가 완료된 데이터를 정상적으로 불러왔습니다. 매출액: {data_after.get('revenue'):,.0f}")
else:
    print("❌ 실패: 공시 이후임에도 데이터를 가져오지 못했습니다.")
print("\n==================================================")

In [ ]:
from data.loader import QuantDataLoader
from datetime import date
import pprint

print("==================================================")
print("🔍 [디버깅] get_quarterly_financials_series 원자료 점검")
print("==================================================\n")

loader = QuantDataLoader()
base_date = date(2026, 7, 31)
test_ticker = '005930'  # 삼성전자

try:
    print(f"⏳ [{test_ticker}] 삼성전자 최근 6개 분기 원자료 조회 중...")
    q_series = loader.get_quarterly_financials_series(test_ticker, base_date, n_quarters=6)
    
    print(f"\n✅ 반환된 리스트 길이 (분기 수): {len(q_series)}")
    
    if len(q_series) == 0:
        print("❌ 빈 리스트가 반환되었습니다. DART API 호출 로직 자체를 점검해야 합니다.")
    else:
        for i, q_data in enumerate(q_series):
            # t=0이 가장 최근 분기, t=5가 5분기 전(직전분기의 전년동기)
            print(f"\n--------------------------------------------------")
            print(f"📅 [t-{i} 분기 데이터]")
            print(f"--------------------------------------------------")
            pprint.pprint(q_data, indent=2, width=80)
            
            # Stage 3에서 필수로 찾는 키값들이 있는지 체크
            required_keys = ['revenue', 'sga', 'gross_profit', 'inventory']
            missing_keys = [key for key in required_keys if key not in q_data or pd.isna(q_data.get(key))]
            if missing_keys:
                print(f"⚠️ 경고: Stage 3 필수 계정 누락 또는 NaN -> {missing_keys}")

except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import yaml
from datetime import date
from data.loader import QuantDataLoader
from pprint import pprint

print("==================================================")
print("🛠️ [단일 종목 정밀 타격] 캐시 바이패스 & Raw Data 디버깅")
print("==================================================\n")

# 1. 파라미터 로드
try:
    with open("config/params.yaml", "r", encoding="utf-8") as f:
        params = yaml.safe_load(f)
except FileNotFoundError:
    params = {} # params가 없어도 loader 단독 테스트가 가능하도록 임시 처리

# 2. 🚨 핵심: 캐시를 강제로 끄고(False) 로더를 새로 인스턴스화 🚨
print("🔄 캐시를 무효화하고 DART 서버에 다이렉트로 요청합니다 (use_cache=False)...")
loader_direct = QuantDataLoader(use_cache=False) 

# 테스트 타겟 설정
test_ticker = '002310'  # 앞선 로그에서 0이 떴던 첫 번째 종목
base_dt = date(2023, 6, 30) # 당시 백테스트 기준일 

print(f"▶️ 타겟 종목: {test_ticker} / 기준일: {base_dt}\n")

try:
    # 3. 로더 호출 (최근 2분기)
    q_series_raw = loader_direct.get_quarterly_financials_series(test_ticker, base_dt, n_quarters=2)
    
    if not q_series_raw:
        print("❌ DART 서버에서 빈 리스트를 반환했습니다. API 키 상태나 종목 코드를 확인하세요.")
    else:
        # 4. 가공된 변수 할당 전, 로더가 뱉어내는 딕셔너리 원본(Raw) 자체를 출력
        print("📊 [로더 반환 Raw 딕셔너리 원본]")
        pprint(q_series_raw[0], width=80, sort_dicts=False)
        print("-" * 50)
        
        # 5. 우리가 필요한 키값이 정상적으로 매핑되어 있는지 확인
        latest = q_series_raw[0]
        print("\n🔍 [파이프라인 매핑 키 검증]")
        print(f" - operating_income (영업이익): {latest.get('operating_income', '⚠️ 키 없음 (KEY_MISSING)')}")
        print(f" - interest_expense (이자비용): {latest.get('interest_expense', '⚠️ 키 없음 (KEY_MISSING)')}")
        print(f" - total_liabilities (총부채):  {latest.get('total_liabilities', '⚠️ 키 없음 (KEY_MISSING)')}")
        print(f" - total_equity (자기자본):     {latest.get('total_equity', '⚠️ 키 없음 (KEY_MISSING)')}")

except Exception as e:
    print(f"❌ 데이터 로드 중 에러 발생: {e}")